In [8]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

def sql_query(query):
    return pd.read_sql(query, engine)

In [ ]:
#estas son todas las clases posibles que hay en las tablas
sql_query("""
    SELECT "Fecha", "Campus", "Vertical" 
    FROM stg_alumnos
    UNION
    SELECT "Fecha", "Campus", "Vertical" 
    FROM stg_claustro
""")

,Fecha,Campus,Vertical
0,2024-02-12,Valencia,Full Stack
1,2024-02-12,Madrid,Data Science
2,2023-09-18,Madrid,Data Science
3,2023-09-18,Valencia,Full Stack
4,2023-09-18,Madrid,Full Stack
5,2024-02-12,Madrid,Full Stack


In [7]:
sql_query('''
    SELECT pos."Fecha" AS fecha_comienzo, c.id_campus, v.id_vertical
    FROM (
        SELECT "Fecha", "Campus", "Vertical" 
        FROM stg_alumnos
        UNION
        SELECT "Fecha", "Campus", "Vertical" 
        FROM stg_claustro
        ) AS pos
    JOIN campus c
    ON c.nombre_campus = pos."Campus"
    JOIN verticales v
    ON v.nombre_vertical = pos."Vertical"

    EXCEPT

    SELECT fecha_comienzo, id_campus, id_vertical FROM promociones
''')

,fecha_comienzo,id_campus,id_vertical


In [ ]:
# #cargando datos de promociones
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO promociones (fecha_comienzo, id_campus, id_vertical)
        SELECT pos."Fecha", c.id_campus, v.id_vertical
            FROM (
                SELECT "Fecha", "Campus", "Vertical" 
                FROM stg_alumnos
                UNION
                SELECT "Fecha", "Campus", "Vertical" 
                FROM stg_claustro
                ) AS pos
            JOIN campus c
            ON c.nombre_campus = pos."Campus"
            JOIN verticales v
            ON v.nombre_vertical = pos."Vertical"
        EXCEPT
        
        SELECT fecha_comienzo, id_campus, id_vertical FROM promociones
    """))

In [12]:
sql_query("""
    SELECT DISTINCT REPLACE(REPLACE("Proyecto", 'Proyecto_', ''), 'FullSatck' ,'FullStack') AS nombre_proyecto, "Vertical"
    FROM stg_resultados
""")

,nombre_proyecto,Vertical
0,Deployment,Data Science
1,EDA,Data Science
2,FullStack,Full Stack
3,Backend,Full Stack
4,FrontEnd,Full Stack
5,HLF,Data Science
6,ML,Data Science
7,WebDev,Full Stack
8,BBDD,Data Science
9,React,Full Stack


In [16]:
sql_query("""
    SELECT proy.nombre_proyecto, v.id_vertical
    FROM (
        SELECT DISTINCT REPLACE(REPLACE("Proyecto", 'Proyecto_', ''), 'FullSatck' ,'FullStack') AS nombre_proyecto, "Vertical"
        FROM stg_resultados
    ) AS proy
    JOIN verticales v
    ON v.nombre_vertical = proy."Vertical"

    EXCEPT
    
    SELECT nombre_proyecto, id_vertical FROM proyectos
""")

,nombre_proyecto,id_vertical


In [ ]:
#cargando datos de proyectos
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO proyectos (nombre_proyecto, id_vertical)
        SELECT proy.nombre_proyecto, v.id_vertical
        FROM (
            SELECT DISTINCT REPLACE(REPLACE("Proyecto", 'Proyecto_', ''), 'FullSatck' ,'FullStack') AS nombre_proyecto, "Vertical"
            FROM stg_resultados
        ) AS proy
        JOIN verticales v
        ON v.nombre_vertical = proy."Vertical"
    
        EXCEPT
        
        SELECT nombre_proyecto, id_vertical FROM proyectos
    """))

In [23]:
sql_query("""
    SELECT a.id_alumno, "Email", "Fecha", "Campus", "Vertical"
    FROM stg_alumnos s
    JOIN alumnos a
    ON s."Email" = a.email 
""")

,id_alumno,Email,Fecha,Campus,Vertical
0,1,Jafet_Casals@gmail.com,2023-09-18,Madrid,Data Science
1,2,Jorge_Manzanares@gmail.com,2023-09-18,Madrid,Data Science
2,3,Onofre_Adadia@gmail.com,2023-09-18,Madrid,Data Science
3,4,Merche_Prada@gmail.com,2023-09-18,Madrid,Data Science
4,5,Pilar_Abella@gmail.com,2023-09-18,Madrid,Data Science
5,6,Leoncio_Tena@gmail.com,2023-09-18,Madrid,Data Science
6,7,Odalys_Torrijos@gmail.com,2023-09-18,Madrid,Data Science
7,8,Eduardo_Caparrós@gmail.com,2023-09-18,Madrid,Data Science
8,9,Ignacio_Goicoechea@gmail.com,2023-09-18,Madrid,Data Science
9,10,Clementina_Santos@gmail.com,2023-09-18,Madrid,Data Science


In [ ]:
sql_query("""
    SELECT f.id_alumno, p.id_promocion
    FROM (
        SELECT a.id_alumno, "Email" as email, "Fecha" AS fecha, "Campus" as campus, "Vertical" as vertical
        FROM stg_alumnos s
        JOIN alumnos a
        ON s."Email" = a.email 
    ) as f
    JOIN campus c
    ON f.campus = c.nombre_campus
    JOIN verticales v
    ON f.vertical = v.nombre_vertical
    JOIN promociones p
    ON p.fecha_comienzo = f.fecha
    AND p.id_campus = c.id_campus
    AND p.id_vertical = v.id_vertical

    EXCEPT

    SELECT id_alumno, id_promocion FROM matriculas

    
""")

,id_alumno,id_promocion


In [ ]:
#cargando datod de matriculas
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO matriculas (id_alumno, id_promocion)
        SELECT f.id_alumno, p.id_promocion
        FROM (
            SELECT a.id_alumno, "Email" as email, "Fecha" AS fecha, "Campus" as campus, "Vertical" as vertical
            FROM stg_alumnos s
            JOIN alumnos a
            ON s."Email" = a.email 
        ) as f
        JOIN campus c
        ON f.campus = c.nombre_campus
        JOIN verticales v
        ON f.vertical = v.nombre_vertical
        JOIN promociones p
        ON p.fecha_comienzo = f.fecha
        AND p.id_campus = c.id_campus
        AND p.id_vertical = v.id_vertical
    
        EXCEPT
    
        SELECT id_alumno, id_promocion FROM matriculas
    """))

In [35]:
sql_query("""
    SELECT p.id_profesor, "Nombre", "Rol", "Modalidad", "Fecha", "Campus", "Vertical"
    FROM stg_claustro sc
    JOIN profesores p
    ON sc."Nombre" = p.nombre_profesor 
""")

,id_profesor,Nombre,Rol,Modalidad,Fecha,Campus,Vertical
0,1,Noa Yáñez,TA,Presencial,2023-09-18,Madrid,Data Science
1,2,Saturnina Benitez,TA,Presencial,2023-09-18,Madrid,Data Science
2,3,Anna Feliu,TA,Presencial,2023-09-18,Madrid,Full Stack
3,4,Rosalva Ayuso,TA,Presencial,2023-09-18,Valencia,Full Stack
4,5,Ana Sofía Ferrer,TA,Presencial,2024-02-12,Valencia,Full Stack
5,6,Angélica Corral,TA,Presencial,2024-02-12,Madrid,Full Stack
6,7,Ariel Lledó,TA,Presencial,2023-09-18,Madrid,Data Science
7,8,Mario Prats,LI,Online,2024-02-12,Valencia,Full Stack
8,9,Luis Ángel Suárez,LI,Online,2023-09-18,Madrid,Full Stack
9,10,María Dolores Diaz,LI,Online,2023-09-18,Madrid,Data Science


In [ ]:
sql_query("""
    SELECT f.id_profesor, pr.id_promocion, m.id_modalidad, f.rol
    FROM(
        SELECT p.id_profesor as id_profesor, "Nombre" as nombre, "Rol" as rol, "Modalidad" as modalidad, "Fecha" as fecha, "Campus" as campus, "Vertical" as vertical
        FROM stg_claustro sc
        JOIN profesores p
        ON sc."Nombre" = p.nombre_profesor
    ) AS f
    JOIN campus c
    ON f.campus = c.nombre_campus
    JOIN verticales v
    ON f.vertical = v.nombre_vertical
    JOIN promociones pr
    ON pr.fecha_comienzo = f.fecha
    AND pr.id_campus = c.id_campus
    AND pr.id_vertical = v.id_vertical 
    JOIN modalidad m
    ON f.modalidad = m.nombre_modalidad

    EXCEPT 
    
    SELECT id_profesor, id_promocion, id_modalidad, rol FROM profesor_promocion
""")

,id_profesor,id_promocion,id_modalidad,rol


In [34]:
#cargando datod de profesor_promocion
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO profesor_promocion (id_profesor, id_promocion, id_modalidad, rol)
        SELECT f.id_profesor, pr.id_promocion, m.id_modalidad, f.rol
        FROM(
            SELECT p.id_profesor as id_profesor, "Nombre" as nombre, "Rol" as rol, "Modalidad" as modalidad, "Fecha" as fecha, "Campus" as campus, "Vertical" as vertical
            FROM stg_claustro sc
            JOIN profesores p
            ON sc."Nombre" = p.nombre_profesor
        ) AS f
        JOIN campus c
        ON f.campus = c.nombre_campus
        JOIN verticales v
        ON f.vertical = v.nombre_vertical
        JOIN promociones pr
        ON pr.fecha_comienzo = f.fecha
        AND pr.id_campus = c.id_campus
        AND pr.id_vertical = v.id_vertical 
        JOIN modalidad m
        ON f.modalidad = m.nombre_modalidad
    
        EXCEPT 
        
        SELECT id_profesor, id_promocion, id_modalidad, rol FROM profesor_promocion
    """))